<a href="https://colab.research.google.com/github/waqas-manzoor5595/Rag_project_/blob/main/Rag_project_on_Dress_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook is designed to be a versatile tool for querying information from any PDF document. By simply uploading a PDF, users can ask questions about its content and receive relevant answers based on the document's text.

For demonstration purposes, this project utilizes the **LPU Dress Code Policy PDF** as an example. This allows us to showcase how the system can accurately extract and synthesize information from a specific institutional document.

This project would be highly beneficial for the university in several ways:
- **Quick Information Retrieval:** Students and staff can quickly get answers to policy-related questions without sifting through long documents.
- **Improved Accessibility:** Makes complex documents more accessible by providing direct answers to queries.
- **Reduced Administrative Burden:** Frees up administrative staff from repeatedly answering common questions.
- **Enhanced Compliance:** Helps ensure better understanding and compliance with university policies by providing clear, on-demand explanations.

In [10]:
!pip install -q langchain langchain-community langchain-groq langchain-text-splitters langchain-huggingface pypdf faiss-cpu sentence-transformers

In [11]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [12]:
from google.colab import files

uploaded = files.upload()  # choose your single PDF here
pdf_path = list(uploaded.keys())[0]

Saving Guidelines for Dress Code and Uniform.pdf to Guidelines for Dress Code and Uniform (1).pdf


In [13]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(pdf_path)
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks")

Split into 15 chunks


In [14]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1038.79it/s]


In [16]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

prompt = ChatPromptTemplate.from_template(
    """Answer the question using only the context below.
If the answer isn't in the context, say you don't know.

Context:
{context}

Question: {question}

Answer:"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [19]:
query = "can i wear sweater in winter?"
answer = rag_chain.invoke(query)
print("Answer:", answer)

Answer: You can wear a sweater in winter, but it should be a plain, formal one. According to the context, "Woolens Plain, formal V-neck Cardigans/pull overs, formal coats, long coats are also allowed."


In [18]:
query = "What is this document about?"

source_docs = retriever.invoke(query)
answer = rag_chain.invoke(query)

print("Answer:", answer)
print("\n--- Sources ---")
for i, doc in enumerate(source_docs, 1):
    print(f"[{i}] Page {doc.metadata.get('page', '?')}: {doc.page_content[:150]}...")

Answer: This document appears to be about the dress code policy for staff members at a university, specifically the Lovely Professional University (LPU), as indicated by the mention of the LPU logo and other university-specific details.

--- Sources ---
[1] Page 6: Page | 7 
 
UMS Path for seeking temporary Dress Code Exemption: 
 
UMS Navigation ---->Division of HR---->Transactions ---- >Request for Temporary Dr...
[2] Page 2: Page | 3 
 
to be in formal attire, as defined in A(vi). 
(iv) Business suits shall be compulsory everyday `for HODs and above in winters 
(15th Octob...
[3] Page 3: Page | 4 
 
Fire Staff Uniform 
 
 
Clothing Male 
Shirt Khaki, Full sleeves 
Trousers Khaki 
Epaulet With LPU Logo and Star 
Beret/Turban with LPU Ba...
[4] Page 2: Duties like Protocol Officer, Liason Officer, etc. 
o All other occasions as may be informed from time to time. 
(C). Category C: To be worn by Securi...
